<a href="https://colab.research.google.com/github/Violetaaa/tfg-learning/blob/main/04_gis_with_python/06_spatial_join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.6 Spatial join (resumen)

Resumen del capítulo *Spatial join* de *Python for Geographic Data Analysis* (Tenkanen et al., pythongis.org), reducido a lo que se usa en el TFG. El libro usa datos de Helsinki; aquí se usan tus datos de Asturias.

**Idea clave:** un *spatial join* es como un `merge` de pandas, pero la condición de unión es espacial (por ejemplo, «el punto está dentro del polígono») en lugar de una columna común. Sirve para **pasar atributos de una capa a otra**. Ejemplo: a cada farmacia le añades el concejo en el que cae.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Folder that contains raw/ and asturias_miteco.gpkg. In Colab, e.g. Path("/content/drive/MyDrive/tfg/datos").
DATA_DIR = Path("/content/drive/MyDrive/tfg/")
CRS_METRES = "EPSG:25830"


def load_layer(pattern):
    """Read the latest raw download matching a pattern and reproject it to metres."""
    path = sorted(DATA_DIR.glob(f"sitpa_raw_data/{pattern}"))[-1]
    return gpd.read_file(path).to_crs(CRS_METRES)


municipalities = gpd.read_file(DATA_DIR / "miteco_data/out/asturias_miteco.gpkg").to_crs(CRS_METRES)
pharmacies = load_layer("farmacias_botiquines__farmacias_*.geojson")


## Los tres parámetros de `sjoin`

`left.sjoin(right, predicate=..., how=...)`

- **Qué capa va a la izquierda:** la que **recibe** los atributos. El resultado tiene la geometría de la capa izquierda.
- **`predicate`:** la relación espacial (`"intersects"` por defecto, `"within"`, `"contains"`…). Ver 6.5.
- **`how`:**
  - `"inner"` (por defecto): solo las filas con coincidencia.
  - `"left"`: todas las filas de la izquierda; las que no encuentran pareja quedan con `NaN`.
  - `"right"`: lo mismo, pero con la capa derecha.

Antes de nada, comprueba que el CRS es el mismo (aquí ya se reproyectó todo a 25830 al cargar).

In [3]:
assert pharmacies.crs == municipalities.crs

# Bring only the columns you need from the right layer
joined = pharmacies.sjoin(
    municipalities[["nombre", "farmacias", "geometry"]],
    predicate="within",
    how="left",
)
joined[["titular", "concejo", "nombre", "index_right"]].head(3)

,titular,concejo,nombre,index_right
0,ALBA FERNANDEZ RODRIGUEZ,AVILES,Avilés,3
1,ANA COVADONGA PARIS GRAIÑO,AVILES,Avilés,3
2,JUAN RODRIGUEZ DE LA FLOR MORO,AVILES,Avilés,3


`nombre` viene del polígono en el que cae cada farmacia. `index_right` es el índice de esa fila en `municipalities`. Si te molesta, se borra con `.drop(columns="index_right")`.

## Comprobar que no se pierden filas
Con `how="left"`, las filas sin pareja tienen `NaN` en las columnas de la derecha. Con `how="inner"` desaparecerían sin avisar. Es una comprobación que conviene hacer siempre. Típicamente fallan los puntos en la costa (un poco mar adentro) o los que caen justo en el borde con `within`.

In [4]:
unmatched = joined[joined["nombre"].isna()]
print(len(pharmacies), "pharmacies |", len(joined), "rows after join |", len(unmatched), "without municipality")

454 pharmacies | 454 rows after join | 0 without municipality


Si una farmacia cayera en **dos** polígonos (porque se solapan, o porque está en el borde y usas `intersects`), aparecería **dos veces**. Por eso conviene comparar `len()` antes y después.

## Uso típico: contar por polígono y validar contra otra fuente
Contar farmacias por concejo según su **ubicación** y compararlo con el recuento del MITECO (2023). Es la validación de datos prevista en el TFG.

In [5]:
counts = joined.groupby("nombre").size().rename("count_sig")

comparison = municipalities.set_index("nombre")[["farmacias"]].join(counts).fillna(0)
comparison = comparison.rename(columns={"farmacias": "count_miteco"})
comparison["diff"] = comparison["count_sig"] - comparison["count_miteco"]

print((comparison["diff"] == 0).sum(), "of", len(comparison), "municipalities match exactly")
comparison[comparison["diff"] != 0]

75 of 78 municipalities match exactly


,count_miteco,count_sig,diff
nombre,,,
Allande,2,1.0,-1.0
Gijón,105,104.0,-1.0
Oviedo,89,90.0,1.0


Las diferencias son de una farmacia en tres concejos. **Hecho:** el SIG es de 2026 y el MITECO de 2023. **Inferencia, sin verificar:** pueden ser aperturas o cierres en ese periodo, o farmacias muy cerca del límite entre Gijón y Oviedo.

## Atributo frente a ubicación
La capa de farmacias ya trae una columna `concejo`. El join por ubicación sirve para comprobarla, pero los nombres no se escriben igual en las dos fuentes (`CANGAS DE NARCEA` frente a `Cangas del Narcea`). **Para unir fuentes distintas, mejor por geometría o por código INE que por nombre.**

In [ ]:
pairs = joined[["concejo", "nombre"]].drop_duplicates()
pairs[pairs["concejo"].str.upper() != pairs["nombre"].str.upper()].head()

,concejo,nombre
0,AVILES,Avilés
6,GOZON,Gozón
9,CASTRILLON,Castrillón
10,GIJON,Gijón
13,CORVERA,Corvera de Asturias


## Lo que se ha omitido del capítulo
- **El join en sentido contrario** (`municipalities.sjoin(pharmacies)`): da una fila por cada par concejo-farmacia. Para contar, el `groupby` sobre las farmacias es más claro.
- **Mapas coloreados del resultado**: se ven en 6.9.
- **Join por cercanía en vez de por contención**: se ve en 6.7 (`sjoin_nearest`).

Documentación: https://geopandas.org/en/stable/docs/user_guide/mergingdata.html#spatial-joins